# Finetuning Granite Speech (LoRA)


In [5]:
!pip install -U "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec peft bitsandbytes


Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 13.3 MB/s  0:00:00


## Setup

In [1]:
import torch
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"
MANIFEST_DIR = Path("asr_finetune")
OUTPUT_DIR = MANIFEST_DIR / "checkpoint"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

model.config.use_cache = False


Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

Model loaded on cuda:0 with dtype torch.bfloat16


## Load the Training and Dev Manifests

In [2]:
import json
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev.jsonl"

def read_manifest(path):
    """Read a JSONL manifest into a list of dicts."""
    with open(path, encoding="UTF-8") as rf:
        return [json.loads(line) for line in rf]

train_data = read_manifest(MANIFEST_TRAIN)
dev_data = read_manifest(MANIFEST_DEV)

print(len(train_data), "train samples,", len(dev_data), "dev samples")
print(train_data[0])

1000 train samples, 100 dev samples
{'audio': '/training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131007-0900-PLENARY-19-en_20131007-21:26:04_7.wav', 'text': 'this could be done by looking more specifically into the suggested fallback clause in article twenty one in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions', 'prompt': 'Vocabulary: article twenty one.'}


## Prompt Construction

In [3]:
import torchaudio

def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform

def build_chat_prompt(user_prompt):
    """Apply the Granite Speech chat template to a transcription request."""
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{user_prompt}"},
    ]
    return processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

DEFAULT_PROMPT = "can you transcribe the speech into a written format?"

def build_user_prompt(sample):
    """Default transcription prompt, with keywords appended when the manifest has them."""
    prompt = sample.get("prompt")
    if prompt:
        return f"{DEFAULT_PROMPT} Keywords: {prompt}"
    return DEFAULT_PROMPT

## LoRA Configuration (PEFT)

In [4]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,259,840 || all params: 2,317,401,436 || trainable%: 0.1838


## Collate Function (Audio + Prompt + Target Text)

In [5]:
def collate_fn(batch):
    wavs = [load_audio(sample["audio"]) for sample in batch]
    prompts = [build_chat_prompt(build_user_prompt(sample)) for sample in batch]

    # Processor pads input_ids, attention_mask, input_features and input_features_mask
    inputs = processor(prompts, wavs, return_tensors="pt", padding=True)   # CPU tensors

    # Append the target transcript as supervision and mask the prompt part
    target_ids = [processor.tokenizer(sample["text"], add_special_tokens=False,
                                      return_tensors="pt").input_ids[0] for sample in batch]
    batch_size, prompt_len = inputs["input_ids"].shape
    max_len = prompt_len + max(len(t) for t in target_ids)
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0

    input_ids = torch.full((batch_size, max_len), pad_id, dtype=inputs["input_ids"].dtype)
    attention_mask = torch.zeros((batch_size, max_len), dtype=inputs["attention_mask"].dtype)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)

    input_ids[:, :prompt_len] = inputs["input_ids"]
    attention_mask[:, :prompt_len] = inputs["attention_mask"]
    for i, t in enumerate(target_ids):
        input_ids[i, prompt_len:prompt_len + len(t)] = t
        attention_mask[i, prompt_len:prompt_len + len(t)] = 1
        labels[i, prompt_len:prompt_len + len(t)] = t   # ignore prompt/audio tokens (-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        **{k: v for k, v in inputs.items() if k not in ("input_ids", "attention_mask")},
    }

batch = collate_fn(train_data[:2])
mask = batch["labels"] != -100
print(processor.tokenizer.decode(batch["labels"][0][mask[0]]))

this could be done by looking more specifically into the suggested fallback clause in article twenty one in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions


## Training Configuration

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR.as_posix(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=1,
    eval_strategy="epoch",
    logging_steps=10,
    logging_first_step=True,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to=[],
    remove_unused_columns=False,
    disable_tqdm=False,
)


## Evaluation (Dev WER)

In [7]:
import jiwer
from tqdm import tqdm

def compute_dev_wer():
    """Greedy-transcribe the dev set with the current model and compute WER"""
    model.eval()
    wers = []
    for sample in tqdm(dev_data):
        wav = load_audio(sample["audio"])
        text_prompt = build_chat_prompt(build_user_prompt(sample))
        inputs = processor(text_prompt, wav, device=model.device, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1,
                                 use_cache=True)
        hyp = processor.tokenizer.decode(
            out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
        wers.append(jiwer.wer(sample["text"], hyp))
    model.train()
    return sum(wers) / len(wers)



In [8]:
print("Dev WER before training:", compute_dev_wer())

100%|██████████| 100/100 [02:14<00:00,  1.35s/it]

Dev WER before training: 0.052007927069935525


## Train

In [9]:
from transformers import Trainer, TrainerCallback


class ConsoleLogCallback(TrainerCallback):
    """Print loss and learning rate at every logging step"""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"step {state.global_step}: {logs}")


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=dev_data,
    data_collator=collate_fn,
    callbacks=[ConsoleLogCallback()],
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,3.740562,3.233810


step 1: {'loss': 6.97576904296875, 'grad_norm': 69.72146606445312, 'learning_rate': 1e-05, 'epoch': 0.004}
step 10: {'loss': 5.665428161621094, 'grad_norm': 30.448692321777344, 'learning_rate': 9.640000000000001e-06, 'epoch': 0.04}
step 20: {'loss': 4.098614883422852, 'grad_norm': 37.7790641784668, 'learning_rate': 9.240000000000001e-06, 'epoch': 0.08}
step 30: {'loss': 3.650558090209961, 'grad_norm': 12.314738273620605, 'learning_rate': 8.84e-06, 'epoch': 0.12}
step 40: {'loss': 10.005311584472656, 'grad_norm': 21.54034423828125, 'learning_rate': 8.44e-06, 'epoch': 0.16}
step 50: {'loss': 5.9262336730957035, 'grad_norm': 15.553240776062012, 'learning_rate': 8.040000000000001e-06, 'epoch': 0.2}
step 60: {'loss': 5.303104400634766, 'grad_norm': 25.99968719482422, 'learning_rate': 7.640000000000001e-06, 'epoch': 0.24}
step 70: {'loss': 7.048250579833985, 'grad_norm': 129.5437469482422, 'learning_rate': 7.24e-06, 'epoch': 0.28}
step 80: {'loss': 4.348228836059571, 'grad_norm': 78.37388610

TrainOutput(global_step=250, training_loss=4.70101676940918, metrics={'train_runtime': 265.5378, 'train_samples_per_second': 3.766, 'train_steps_per_second': 0.941, 'total_flos': 2356146659044008.0, 'train_loss': 4.70101676940918, 'epoch': 1.0})

In [10]:
print("Dev WER AFTER training:", compute_dev_wer())

100%|██████████| 100/100 [01:56<00:00,  1.17s/it]

Dev WER AFTER training: 0.07622029843126138


## Write Out the Checkpoint (LoRA Merged into the Base Model)

In [18]:
merged_model = model.merge_and_unload()   # merge LoRA weights into the base model
merged_model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Checkpoint written to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint written to asr_finetune/checkpoint


In [8]:
sample = train_data[0]
wav = load_audio(sample["audio"])
inputs = processor(build_chat_prompt(build_user_prompt(sample)), wav,
                   return_tensors="pt").to(model.device, model.dtype)

target = processor.tokenizer(sample["text"], add_special_tokens=False,
                             return_tensors="pt").input_ids.to(model.device)

full_ids = torch.cat([inputs["input_ids"][0], target[0]]).unsqueeze(0)   # clean token ids
labels = full_ids.clone()
labels[:, : inputs["input_ids"].shape[1]] = -100                          # mask prompt/audio

attention_mask = torch.cat([
    inputs["attention_mask"][0],
    torch.ones(target.shape[1], dtype=inputs["attention_mask"].dtype, device=model.device),
]).unsqueeze(0)

with torch.inference_mode():
    out1 = model(
        input_ids=full_ids,
        attention_mask=attention_mask,
        labels=labels,
        **{k: v for k, v in inputs.items() if k not in ("input_ids", "attention_mask")},
    )
print("single-sample loss:", out1.loss.item())

# Way 2: through collate_fn (batched)
batch = collate_fn(train_data[:2])
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.inference_mode():
    out2 = model(**batch)
print("collate loss:", out2.loss.item())

single-sample loss: 0.00110626220703125
collate loss: 0.498046875
